# Assignment 7 - Consensus and Forks Lab

**Module:** BLCH9X2
**Module file:** `consensus.py` (Assignment 7)

**Goal:** Simulate nodes A, B and C with competing tips, validate a chain before comparing it to others, resolve a fork using the longest-chain / heaviest-work rule, compute confirmation depth for a R10,000 remittance, and document the required edge cases (ties, delayed block arrival, confirmation depth).


## 0. Learning map

| Section | What you build | Part A requirement |
|---|---|---|
| 1 | Minimal block helpers and genesis block | Supports (a) and (b) |
| 2 | `verify_chain` - validate before compare | Supports (b) |
| 3 | `Node` class for A, B and C | (a) three nodes |
| 4 | Competing tips / create a fork | (b) create a fork |
| 5 | `chain_work` - longest vs heaviest-work | (b) resolution rule |
| 6 | `tie_break` and `resolve_fork` | (b) resolve fork, (c) ties |
| 7 | Extend a branch and `sync_nodes` | (b) resolve fork |
| 8 | Delayed arrival and reorganisation | (c) delayed block arrival |
| 9 | Confirmation depth k = H - h + 1 | (c) confirmation depth |
| 10 | Remittance scenario (R10,000) | (c) R10,000 payment |
| 11 | `demo_fork_scenario` end-to-end demo | (a), (b) combined |



## 1. Minimal block and chain helpers


In [1]:
from __future__ import annotations

import hashlib
import json
import time
from copy import deepcopy
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

Block = dict[str, Any]
Chain = list[Block]

DIFFICULTY = 2  # leading hex zeros - toy classroom setting


def _sha(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def _header_hash(index, timestamp, txs, previous_hash, nonce) -> str:
    payload = json.dumps(
        {
            "index": index,
            "timestamp": timestamp,
            "transactions": txs,
            "previous_hash": previous_hash,
            "nonce": nonce,
        },
        sort_keys=True,
    )
    return _sha(payload)


def mine_block(
    index, txs, previous_hash, difficulty: int = DIFFICULTY
) -> Block:
    """Proof-of-work search until the hash has `difficulty` leading hex zeros."""
    timestamp = time.time()
    nonce = 0
    while True:
        h = _header_hash(index, timestamp, txs, previous_hash, nonce)
        if h.startswith("0" * difficulty):
            return {
                "index": index,
                "timestamp": timestamp,
                "transactions": txs,
                "previous_hash": previous_hash,
                "nonce": nonce,
                "hash": h,
                "difficulty": difficulty,
            }
        nonce += 1


def make_genesis() -> Block:
    return mine_block(0, [{"note": "genesis"}], "0" * 64)


genesis = make_genesis()
print("Genesis hash:", genesis["hash"][:16], "...")
print("Difficulty:", genesis["difficulty"], "| nonce:", genesis["nonce"])

Genesis hash: 0034d7984287a86f ...
Difficulty: 2 | nonce: 525


## 2. Validate before compare - `verify_chain`

**Invariant:** a longer invalid chain must never win fork choice.

Checks, in order:
1. recomputed header hash matches the stored `hash`;
2. proof-of-work / difficulty prefix;
3. parent links (`previous_hash`);
4. sequential `index`.

Only chains that pass these checks are allowed to compete inside `resolve_fork`.

In [2]:
def verify_chain(chain: Chain, difficulty: int = DIFFICULTY) -> bool:
    """Return True iff hashes, parent links, and PoW difficulty checks pass."""
    if not chain:
        return False
    for i, block in enumerate(chain):
        expected = _header_hash(
            block["index"],
            block["timestamp"],
            block["transactions"],
            block["previous_hash"],
            block["nonce"],
        )
        if block.get("hash") != expected:
            return False
        diff = int(block.get("difficulty", difficulty))
        if not str(block["hash"]).startswith("0" * diff):
            return False
        if i == 0:
            continue
        if block["previous_hash"] != chain[i - 1]["hash"]:
            return False
        if block["index"] != i:
            return False
    return True


print("Genesis alone verifies:", verify_chain([genesis]))

# Junk dicts must not pass - even if the list is "long"
junk = [deepcopy(genesis)] + [
    {"index": i, "hash": "ff" * 32, "previous_hash": "x", "nonce": 0,
     "timestamp": 0, "transactions": [], "difficulty": DIFFICULTY}
    for i in range(1, 6)
]
print("Long junk list verifies:", verify_chain(junk), "(must be False)")

Genesis alone verifies: True
Long junk list verifies: False (must be False)


## 3. Creating Nodes A, B, C 
- Three nodes are created below, each starting from the same genesis block. In-process simulation is utilised in notebook

In [3]:
def chain_work(chain: Chain) -> int:
    """Heaviest-work score: sum of per-block difficulties.

    Equals length ranking when every block shares the same difficulty.
    """
    total = 0
    for block in chain:
        total += int(block.get("difficulty", DIFFICULTY))
    return total


@dataclass
class Node:
    """A simulated peer holding its own view of the chain."""

    name: str
    chain: Chain = field(default_factory=list)

    def tip(self) -> Block:
        if not self.chain:
            raise ValueError(f"node {self.name} has an empty chain")
        return self.chain[-1]

    def height(self) -> int:
        return len(self.chain) - 1 if self.chain else -1

    def summary(self) -> str:
        tip_short = self.tip()["hash"][:10]
        return (
            f"{self.name}: height={self.height()} tip={tip_short}... "
            f"valid={verify_chain(self.chain)} work={chain_work(self.chain)}"
        )


A = Node("A", [deepcopy(genesis)])
B = Node("B", [deepcopy(genesis)])
C = Node("C", [deepcopy(genesis)])

for n in (A, B, C):
    print(n.summary())
print("Shared tip?", A.tip()["hash"] == B.tip()["hash"] == C.tip()["hash"])

A: height=0 tip=0034d79842... valid=True work=2
B: height=0 tip=0034d79842... valid=True work=2
C: height=0 tip=0034d79842... valid=True work=2
Shared tip? True


## 4. Create a fork

A and B mine different block-1 payloads extending the same genesis parent:
- A includes a R10,000 payment to Alice (`pay-alice`);
- B includes a R10,000 payment to Bob (`pay-bob`).
- C stays at genesis.

In [4]:
tx_alice = [
    {
        "id": "pay-alice",
        "sender": "bank",
        "recipient": "Alice",
        "amount": 10000,
        "currency": "ZAR",
    }
]
tx_bob = [
    {
        "id": "pay-bob",
        "sender": "bank",
        "recipient": "Bob",
        "amount": 10000,
        "currency": "ZAR",
    }
]

block_A1 = mine_block(1, tx_alice, A.tip()["hash"])
block_B1 = mine_block(1, tx_bob, B.tip()["hash"])

A.chain.append(block_A1)
B.chain.append(block_B1)
# C still at genesis only

print("A and B tips differ?", A.tip()["hash"] != B.tip()["hash"])
print("Both extensions valid?",
      verify_chain(A.chain), "|", verify_chain(B.chain))
print("Same parent (genesis)?",
      A.chain[1]["previous_hash"] == B.chain[1]["previous_hash"] == genesis["hash"])
for n in (A, B, C):
    print(" ", n.summary())

print("\nBoard sketch:")
print("  A: G -> 1_A (pay-alice)")
print("  B: G -> 1_B (pay-bob)")
print("  C: G")

A and B tips differ? True
Both extensions valid? True | True
Same parent (genesis)? True
  A: height=1 tip=00ed0b3d99... valid=True work=4
  B: height=1 tip=00949316e9... valid=True work=4
  C: height=0 tip=0034d79842... valid=True work=2

Board sketch:
  A: G -> 1_A (pay-alice)
  B: G -> 1_B (pay-bob)
  C: G


## 5. Scoring - longest-chain vs heaviest-work

For this lab, `chain_work` (cumulative difficulty) is the rule implemented in `resolve_fork`.

In [14]:
def chain_length(chain: Chain) -> int:
    """Longest-chain score (for comparison against chain_work)."""
    return len(chain)


print(f"A: length={chain_length(A.chain)} work={chain_work(A.chain)}")
print(f"B: length={chain_length(B.chain)} work={chain_work(B.chain)}")
print(f"C: length={chain_length(C.chain)} work={chain_work(C.chain)}")
print("Equal scores at the fork tip -> need a tie policy.")

A: length=6 work=12
B: length=3 work=6
C: length=3 work=6
Equal scores at the fork tip -> need a tie policy.


## 6. Tie-break and `resolve_fork`

Resolution steps, matching the course API:
1. filter the candidate chains using `verifier` (default `verify_chain`)
2. maximise `chain_work`
3. on a tie, call `tie_break` which select the chain with the lexicographically smallest tip hash. This makes fork resolution deterministic and independent of input order
4. Return a deep copy of the winning chain, so callers cannot mutate node state by accident.


In [ ]:
def tie_break(chains: list[Chain]) -> Chain:
    """Lexicographically smaller tip hash """
    return min(chains, key=lambda c: c[-1]["hash"])


def resolve_fork(
    chains: list[Chain], verifier: Callable[[Chain], bool] | None = None
) -> Chain:
    """Among valid chains, return a deep copy of the winning chain."""
    verify = verifier or verify_chain
    valid = [c for c in chains if verify(c)]
    if not valid:
        raise ValueError("no valid chain")
    best = max(chain_work(c) for c in valid)
    tied = [c for c in valid if chain_work(c) == best]
    chosen = tied[0] if len(tied) == 1 else tie_break(tied)
    return deepcopy(chosen)


# Equal work at height 1 -> deterministic tie-break
winner_tied = resolve_fork([A.chain, B.chain, C.chain])
w1 = resolve_fork([A.chain, B.chain])
w2 = resolve_fork([B.chain, A.chain])
print("Tie at equal work - tip chosen:", winner_tied[-1]["hash"][:16], "...")
print("Order-independent?", w1[-1]["hash"] == w2[-1]["hash"])
print("Contains pay-alice?",
      any(tx.get("id") == "pay-alice" for tx in winner_tied[-1].get("transactions", [])))
print("Contains pay-bob?",
      any(tx.get("id") == "pay-bob" for tx in winner_tied[-1].get("transactions", [])))

# Invalid chain must never win even if longer
invalid_long = deepcopy(A.chain)
fake = dict(invalid_long[-1])
fake["hash"] = "00" + "f" * 62  # looks PoW-ish but the hash does not recompute
invalid_long[-1] = fake
invalid_long.append(
    {"index": 2, "timestamp": 0, "transactions": [], "previous_hash": fake["hash"],
     "nonce": 0, "hash": "00" + "e" * 62, "difficulty": DIFFICULTY}
)
resolved = resolve_fork([invalid_long, B.chain])
print("Invalid longer chain rejected; winner height:", resolved[-1]["index"])
print("Winner valid?", verify_chain(resolved))

Tie at equal work - tip chosen: 00949316e93460cc ...
Order-independent? True
Contains pay-alice? False
Contains pay-bob? True
Invalid longer chain rejected; winner height: 1
Winner valid? True


## 7. Break the tie - extend one branch and `sync_nodes`

The divergent state is restored, block 2 is mined on A's branch, and all nodes are synced to the heaviest (equivalently longest, under uniform difficulty) valid chain.

`sync_nodes` resolves once and deep-copies the winner onto every node, so after this cell A, B and C should share one canonical tip.

In [19]:
def sync_nodes(nodes: list[Node]) -> Chain:
    """Resolve all node chains and set every node to the winner (deep copy)."""
    winner = resolve_fork([n.chain for n in nodes])
    for node in nodes:
        node.chain = deepcopy(winner)
    return winner


# Restore divergent state for the demo
A.chain = [deepcopy(genesis), deepcopy(block_A1)]
B.chain = [deepcopy(genesis), deepcopy(block_B1)]
C.chain = [deepcopy(genesis)]

block_A2 = mine_block(2, [{"note": "extend-A"}], A.tip()["hash"])
A.chain.append(block_A2)

print("Before sync:")
for n in (A, B, C):
    print(" ", n.summary())

heights_before = {"A": A.height(), "B": B.height(), "C": C.height()}
canonical = sync_nodes([A, B, C])
print('='*57)
print("After sync (heaviest / longest under uniform difficulty):")
print('='*57)
for n in (A, B, C):
    print(" ", n.summary())
print("Heights before:", heights_before)
print("Shared tip after:", A.tip()["hash"][:16], "...")
print("Canonical payment id:", A.chain[1]["transactions"][0].get("id"))

Before sync:
  A: height=2 tip=003839fc75... valid=True work=6
  B: height=1 tip=00949316e9... valid=True work=4
  C: height=0 tip=0034d79842... valid=True work=2
After sync (heaviest / longest under uniform difficulty):
  A: height=2 tip=003839fc75... valid=True work=6
  B: height=2 tip=003839fc75... valid=True work=6
  C: height=2 tip=003839fc75... valid=True work=6
Heights before: {'A': 2, 'B': 1, 'C': 0}
Shared tip after: 003839fc757b91dc ...
Canonical payment id: pay-alice


## 8. Delayed arrival and reorganisation 

A delayed block is one that arrives late at a node, because of network latency or message reordering.

Walkthrough for node C:
1. C hears B's tip first and adopts B's branch;
2. C later hears A's longer / heavier chain and reorganises to A's tip.


In [23]:
# Rebuild a fresh C and a preserved copy of A's winning chain
chain_B_short = [deepcopy(genesis), deepcopy(block_B1)]
chain_A_long = deepcopy(canonical)  # G -> 1_A -> 2_A

C.chain = [deepcopy(genesis)]
print("C start:", C.summary())

# First rumour: B's tip arrives
C.chain = resolve_fork([C.chain, chain_B_short])
print('='*40)
print("C after hearing B:")
print('='*40)

print("C after hearing B:", C.summary())
print("  local payment id:", C.chain[1]["transactions"][0].get("id"))

# Later: A's heavier chain arrives -> reorganisation
before_tip = C.tip()["hash"]
C.chain = resolve_fork([C.chain, chain_A_long])
after_tip = C.tip()["hash"]
print('='*40)
print("C after hearing A (heavier):")
print('='*40)
print("C after hearing A:", C.summary())
print("  reorganisation occurred?", before_tip != after_tip)
print("  local payment id now:", C.chain[1]["transactions"][0].get("id"))

C start: C: height=0 tip=0034d79842... valid=True work=2
C after hearing B:
C after hearing B: C: height=1 tip=00949316e9... valid=True work=4
  local payment id: pay-bob
C after hearing A (heavier):
C after hearing A: C: height=2 tip=003839fc75... valid=True work=6
  reorganisation occurred? True
  local payment id now: pay-alice


## 9. Confirmation depth

If payment $P$ is included at height $h$ and the current tip is at height $H$, the confirmation depth is:

$$
k = H - h + 1
$$

- $k = 1$ means $P$ is in the tip block, with nothing mined on top of it yet;
- $k = 6$ means five further blocks have been mined on top of the block that included $P$.

A larger $k$ means more accumulated proof-of-work would need to be redone to reverse the payment through a reorganisation, so the payment is treated as safer to rely on.

In [9]:
def find_tx_height(chain: Chain, tx_id: str) -> int | None:
    """Return the block index containing tx_id, or None."""
    for block in chain:
        for tx in block.get("transactions", []):
            if tx.get("id") == tx_id:
                return int(block["index"])
    return None


def confirmations(chain: Chain, tx_id: str) -> int:
    """Confirmation depth k for a transaction id.

    If included at height h and tip height is H, k = H - h + 1.
    Return 0 if the transaction is absent.
    """
    h = find_tx_height(chain, tx_id)
    if h is None or not chain:
        return 0
    tip_h = int(chain[-1]["index"])
    return tip_h - h + 1


# Arithmetic drill: h=4, H=9 -> k=6
assert (9 - 4 + 1) == 6

pay_id = "pay-alice"
print(f"{pay_id!r} height:", find_tx_height(A.chain, pay_id))
print(f"{pay_id!r} confirmations at current tip:", confirmations(A.chain, pay_id))
print("'pay-bob' confirmations (orphaned):", confirmations(A.chain, "pay-bob"))

# Pad the tip to deepen confirmations
for i in range(A.height() + 1, 6):
    nxt = mine_block(i, [{"note": f"padding-{i}"}], A.tip()["hash"])
    A.chain.append(nxt)

print("After padding to height", A.height(),
      "-> confirmations:", confirmations(A.chain, pay_id))
print("Missing tx ->", confirmations(A.chain, "missing"))

'pay-alice' height: 1
'pay-alice' confirmations at current tip: 2
'pay-bob' confirmations (orphaned): 0
After padding to height 5 -> confirmations: 5
Missing tx -> 0


## 10. Remittance confirmation scenario (R10,000)

Alice's remittance of R10,000 sits in the canonical block at height 1. The cell below checks the payment against three illustrative confirmation policies.

- What confirmation depth $k$ would be required before treating this remittance as settled, for internal risk purposes?
- A coffee-shop style policy ($k = 1$) offers a fast user experience but carries higher reorganisation exposure. An OTC-desk style policy ($k = 6$) is slower but lowers that exposure. Both are policies applied to the same underlying ledger.
- If the cost of renting enough hashpower to reorganise $k$ blocks is lower than R10,000, the chosen policy is inconsistent with the actual threat it is meant to protect against.


In [28]:
REQUIRED_K_COFFEE = 1   # fast user experience, high reorganisation exposure
REQUIRED_K_OTC = 6      # slower user experience, lower reorganisation risk
REQUIRED_K_CLASS_POLICY = 3  # example justified policy for the toy chain

k_now = confirmations(A.chain, "pay-alice")


def settled_for_policy(chain: Chain, tx_id: str, required_k: int) -> bool:
    return confirmations(chain, tx_id) >= required_k


print(f"pay-alice confirmations now: {k_now}")
print(f"  coffee-shop (k>={REQUIRED_K_COFFEE}):",
      settled_for_policy(A.chain, "pay-alice", REQUIRED_K_COFFEE))
print(f"  class policy (k>={REQUIRED_K_CLASS_POLICY}):",
      settled_for_policy(A.chain, "pay-alice", REQUIRED_K_CLASS_POLICY))
print(f"  OTC desk (k>={REQUIRED_K_OTC}):",
      settled_for_policy(A.chain, "pay-alice", REQUIRED_K_OTC))

print("\nQualitative check for the report:")
print("  Ticket size: R10,000")
print("  Toy difficulty:", DIFFICULTY, "(not production security)")
print("  Chosen classroom k:", REQUIRED_K_CLASS_POLICY)
print("  Justification: a reorganisation must redo k blocks of work; " \
          "\n                 a larger ticket size usually justifies a larger k.")

pay-alice confirmations now: 2
  coffee-shop (k>=1): True
  class policy (k>=3): False
  OTC desk (k>=6): False

Qualitative check for the report:
  Ticket size: R10,000
  Toy difficulty: 2 (not production security)
  Chosen classroom k: 3
  Justification: a reorganisation must redo k blocks of work; 
                 a larger ticket size usually justifies a larger k.


## 11. End-to-end demo - demo_fork_scenario


In [11]:
def demo_fork_scenario() -> dict[str, Any]:
    """Build genesis, fork A/B, extend A, sync, return a summary dict."""
    g = mine_block(0, [{"note": "genesis"}], "0" * 64)
    a = Node("A", [deepcopy(g)])
    b = Node("B", [deepcopy(g)])
    c = Node("C", [deepcopy(g)])

    a.chain.append(
        mine_block(1, [{"id": "pay-alice", "amount": 10000}], a.tip()["hash"])
    )
    b.chain.append(
        mine_block(1, [{"id": "pay-bob", "amount": 10000}], b.tip()["hash"])
    )
    a.chain.append(mine_block(2, [{"note": "extend-A"}], a.tip()["hash"]))

    before = {"A": a.height(), "B": b.height(), "C": c.height()}
    sync_nodes([a, b, c])
    return {
        "heights_before": before,
        "tip_after": a.tip()["hash"],
        "canonical_tx": a.chain[1]["transactions"][0]["id"],
        "confirmations_alice": confirmations(a.chain, "pay-alice"),
        "confirmations_bob": confirmations(a.chain, "pay-bob"),
    }


summary = demo_fork_scenario()
print(json.dumps(summary, indent=2))
assert summary["canonical_tx"] == "pay-alice"
assert summary["confirmations_bob"] == 0
assert summary["confirmations_alice"] >= 1
print("demo_fork_scenario checks passed.")

{
  "heights_before": {
    "A": 2,
    "B": 1,
    "C": 0
  },
  "tip_after": "0050cb6102f072030c2291971dd0ada6b407e14f89eeb57d744385747d8e69e9",
  "canonical_tx": "pay-alice",
  "confirmations_alice": 2,
  "confirmations_bob": 0
}
demo_fork_scenario checks passed.
